In [ ]:
# yolov8 프로젝트 다운
!git clone https://github.com/autogyro/yolo-V8.git

Cloning into 'yolo-V8'...
remote: Enumerating objects: 2723, done.
remote: Total 2723 (delta 0), reused 0 (delta 0), pack-reused 2723 (from 1)
Receiving objects: 100% (2723/2723), 1.41 MiB | 10.10 MiB/s, done.
Resolving deltas: 100% (1854/1854), done.
Updating files: 100% (161/161), done.


In [ ]:
# 프로젝트 경로 이동
%cd /content/drive/MyDrive/yolo-V8

/content/drive/MyDrive/yolo-V8


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics
!pip install -r requirements.txt

In [ ]:
# yolov8n.pt 파일이 생선된 것 확인
!yolo predict


WARNING ⚠️ 'model' argument is missing. Using default 'model=yolo26n.pt'.
WARNING ⚠️ 'source' argument is missing. Using default 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

image 1/2 /usr/local/lib/python3.12/dist-packages/ultralytics/assets/bus.jpg: 640x480 4 persons, 1 bus, 91.3ms
image 2/2 /usr/local/lib/python3.12/dist-packages/ultralytics/assets/zidane.jpg: 384x640 2 persons, 1 tie, 90.9ms
Speed: 6.2ms preprocess, 91.1ms inference, 10.8ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/drive/MyDrive/yolo-V8/runs/detect/predict-3
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
# 데이터셋 다운로드
!pwd

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="***************")
project = rf.workspace("joseph-nelson").project("hard-hat-workers")
version = project.version(1)
dataset = version.download("yolov8")

/content/drive/MyDrive/yolo-V8
loading Roboflow workspace...
loading Roboflow project...


In [ ]:
# train 폴더와 test 폴더 생성 확인
!pwd
from glob import glob

train_img_list = glob('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/train/images/*.jpg')
train_txt_list = glob('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/train/labels_2/*.txt')
test_img_list = glob('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/test/images/*.jpg')
test_txt_list = glob('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/test/labels_2/*.txt')

print(len(train_img_list), len(train_txt_list))
print(len(test_img_list), len(test_txt_list))

/content/drive/MyDrive/yolo-V8
5269 2306
1766 1766


In [ ]:
# train 데이터와 val 데이터 나누기
from sklearn.model_selection import train_test_split

train_img_list, val_img_list = train_test_split(train_img_list, test_size=0.2, random_state=2000)

print(len(train_img_list), len(val_img_list))

4215 1054


In [ ]:
# 파일 쓰기
with open('/content/drive/MyDrive/yolo-V8/train.txt', 'w') as f:
  f.write('\n'.join(train_img_list) + '\n')

with open('/content/drive/MyDrive/yolo-V8/val.txt', 'w') as f:
  f.write('\n'.join(val_img_list) + '\n')

In [ ]:
# yaml 파일 수정
import yaml

with open('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/data.yaml', 'r') as f:
  data = yaml.full_load(f) #yaml.load -> yaml.full_load

print(data)

data['train'] = '/content/drive/MyDrive/yolo-V8/train.txt'
data['val'] = '/content/drive/MyDrive/yolo-V8/val.txt'

with open('/content/drive/MyDrive/yolo-V8/Hard-Hat-Workers-1/data.yaml', 'w') as f:
  yaml.dump(data, f)

{'names': ['head', 'helmet', 'person'], 'nc': 3, 'roboflow': {'license': 'Public Domain', 'project': 'hard-hat-workers', 'url': 'https://universe.roboflow.com/joseph-nelson/hard-hat-workers/dataset/1', 'version': 1, 'workspace': 'joseph-nelson'}, 'test': '../test/images', 'train': '/content/drive/MyDrive/yolo-V8/train.txt', 'val': '/content/drive/MyDrive/yolo-V8/val.txt'}


In [ ]:
!pwd
!yolo detect train data=data.yaml model=yolov8n.pt epochs=30 lr0=0.01

/content/drive/MyDrive/yolo-V8
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 